# 03 Exploratory Data Analysis

Creates data quality checks, summary statistics, and EDA plots for the synthetic IAQ dataset.

**Run order:** this notebook is part of the AirAware workflow and uses the shared repository folders: `data/`, `data/processed/`, and `outputs/`.

"""
IAQ Synthetic Dataset – Exploratory Data Analysis Script
=========================================================
Usage:
    python eda_iaq.py

Outputs:
    eda_report/  – one PNG per section + a summary text report
"""

In [ ]:
#import necessary libraries
import os
import warnings
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import matplotlib.ticker as mticker

warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# Project paths
# ============================================================
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(exist_ok=True)
PROCESSED_DATA_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)


In [ ]:
# -------------------------------
# Set paths 
# --------------------------------
DATA_PATH = DATA_DIR / "synthetic_air_quality_dataset_final.csv"
EF_PATH   = DATA_DIR / "unified_emission_factors_literature.csv"
OUT_DIR   = OUTPUTS_DIR / "eda_report"
OUT_DIR.mkdir(exist_ok=True)

REPORT_LINES = []   # collects text for the summary report

# -----------
# Set styles
# -----------
BLUE   = "#2E75B6"
NAVY   = "#1F3864"
GREEN  = "#70AD47"
AMBER  = "#FFC000"
RED    = "#FF4444"
LIGHT  = "#EEF3F9"
GREY   = "#BFBFBF"

plt.rcParams.update({
    "font.family":     "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.facecolor":     LIGHT,
    "figure.facecolor":   "white",
    "axes.titlesize":     11,
    "axes.labelsize":     9,
    "xtick.labelsize":    8,
    "ytick.labelsize":    8,
    "axes.grid":          True,
    "grid.color":         "white",
    "grid.linewidth":     1.2,
})

# ----------
# helpers
# ----------

def log(msg, title=False):
    if title:
        REPORT_LINES.append("\n" + "="*60)
        REPORT_LINES.append(msg)
        REPORT_LINES.append("="*60)
    else:
        REPORT_LINES.append(msg)

def save(fig, name):
    p = OUT_DIR / name
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved → {p}")

def status_badge(ok):
    return "✅ PASS" if ok else "❌ FAIL"

# ---------------------------------
# load data and initial processing
# ---------------------------------
print("Loading data …")
df = pd.read_csv(DATA_PATH, parse_dates=["start_time_local", "end_time_local"])
ef = pd.read_csv(EF_PATH)
df["hour"] = df["start_time_local"].dt.hour

#--------------------------------------
# SECTION 1 – DATA QUALITY & STRUCTURE
# --------------------------------------
print("\n[1/12] Data Quality & Structure")
log("SECTION 1 – DATA QUALITY & STRUCTURE", title=True)
log(f"Rows        : {len(df):,}")
log(f"Columns     : {df.shape[1]}")
log(f"Duplicates  : {df.duplicated().sum()}")

missing = df.isnull().sum()
missing = missing[missing > 0]
log(f"\nMissing values:")
if missing.empty:
    log("  None")
else:
    for col, n in missing.items():
        log(f"  {col}: {n} ({100*n/len(df):.1f}%) — all at hour=23 (boundary label bug)")

log(f"\nDtypes:\n{df.dtypes.to_string()}")

# figure: missing values bar + dtype heatmap
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Section 1 – Data Quality & Structure", fontsize=13, fontweight="bold", color=NAVY)

# missing bar
miss_all = df.isnull().sum().sort_values(ascending=True)
miss_all = miss_all[miss_all > 0] if miss_all.any() else pd.Series({"No missing values": 0})
colors = [RED if v > 0 else GREEN for v in miss_all.values]
axes[0].barh(miss_all.index, miss_all.values, color=colors)
axes[0].set_title("Missing Value Counts")
axes[0].set_xlabel("Count")
for i, v in enumerate(miss_all.values):
    axes[0].text(v + 1, i, f"{v}", va="center", fontsize=8)

# dtype summary
dtype_counts = df.dtypes.astype(str).value_counts()
wedge_cols = [BLUE, GREEN, AMBER, RED][:len(dtype_counts)]
axes[1].pie(dtype_counts.values, labels=dtype_counts.index, colors=wedge_cols,
            autopct="%1.0f%%", startangle=140)
axes[1].set_title("Column Data Types Distribution")

plt.tight_layout()
save(fig, "01_data_quality.png")

# -------------------------------
# SECTION 2 – TIME & DURATION
# -------------------------------
print("[2/12] Time & Duration")
log("SECTION 2 – TIME & DURATION", title=True)

dur_min = df.duration_min.min()
dur_max = df.duration_min.max()
end_ok  = (df.end_time_local > df.start_time_local).all()
hours   = df.hour.nunique()

log(f"Duration range : {dur_min} – {dur_max} min  (expected 5–120)")
log(f"  Durations < 5  : {(df.duration_min < 5).sum()}")
log(f"  Durations > 120: {(df.duration_min > 120).sum()}")
log(f"  ⚠ Max is {dur_max} — never reaches 60–120 min range")
log(f"end_time > start_time always: {status_badge(end_ok)}")
log(f"Unique hours covered: {hours}/24  {status_badge(hours==24)}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Section 2 – Time & Duration", fontsize=13, fontweight="bold", color=NAVY)

# Duration histogram
axes[0].hist(df.duration_min, bins=30, color=BLUE, edgecolor="white")
axes[0].axvline(120, color=RED, linestyle="--", label="Max expected (120)")
axes[0].axvline(dur_max, color=AMBER, linestyle="--", label=f"Actual max ({dur_max})")
axes[0].set_title("Duration Distribution")
axes[0].set_xlabel("Minutes")
axes[0].legend(fontsize=7)

# Hourly session count
hourly = df.groupby("hour").size()
axes[1].bar(hourly.index, hourly.values, color=BLUE)
axes[1].set_title("Sessions per Hour of Day")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Count")

# time_of_day distribution
tod = df.time_of_day.value_counts()
axes[2].bar(tod.index, tod.values, color=[BLUE, GREEN, AMBER, NAVY][:len(tod)])
axes[2].set_title("time_of_day Distribution\n(416 missing at hour=23)")
axes[2].set_xlabel("Period")
axes[2].set_ylabel("Count")

plt.tight_layout()
save(fig, "02_time_duration.png")

# -----------------------------------
# SECTION 3 – CATEGORICAL VARIABLES
# -----------------------------------
print("[3/12] Categorical Variables")
log("SECTION 3 – CATEGORICAL VARIABLES", title=True)

for col in ["activity_type", "motion_type", "environment_type", "cooking_method"]:
    vc = df[col].value_counts()
    log(f"\n{col}:\n{vc.to_string()}")

log(f"\nmotion_type   : only 'stationary' — no diversity ⚠")
log(f"environment_type: only 'indoor'      — no diversity ⚠")
log(f"'oven' present in emission factors but absent from dataset ⚠")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Section 3 – Categorical Variables", fontsize=13, fontweight="bold", color=NAVY)
axes = axes.flatten()

cats = ["activity_type", "cooking_method", "motion_type", "environment_type"]
for ax, col in zip(axes, cats):
    vc = df[col].value_counts()
    bars = ax.bar(vc.index, vc.values, color=BLUE)
    ax.set_title(col)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=15)
    if col in ["motion_type", "environment_type"]:
        ax.set_facecolor("#FFF2CC")
        ax.set_title(f"{col}\n⚠ No diversity", color="#B8860B")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f"{bar.get_height():,}", ha="center", fontsize=8)

plt.tight_layout()
save(fig, "03_categorical.png")

# ---------------------------------------
# SECTION 4 – OUTDOOR POLLUTION BASELINE
# ---------------------------------------
print("[4/12] Outdoor Pollution Baseline")
log("SECTION 4 – OUTDOOR POLLUTION BASELINE", title=True)

out_cols = ["outdoor_pm25", "outdoor_pm10", "outdoor_no2", "outdoor_o3", "outdoor_so2"]
neg_counts = (df[out_cols] < 0).sum()
pm10_ok    = (df.outdoor_pm10 >= df.outdoor_pm25).all()
o3_no2_r   = df[["outdoor_o3", "outdoor_no2"]].corr().iloc[0, 1]

log(f"Negative values: {neg_counts.to_dict()}  {status_badge(neg_counts.sum()==0)}")
log(f"PM10 >= PM2.5 always: {status_badge(pm10_ok)}")
log(f"O3 ~ NO2 correlation: {o3_no2_r:.3f}  {status_badge(o3_no2_r < 0)}")
log(f"\nDescriptive stats:\n{df[out_cols].describe().round(2).to_string()}")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Section 4 – Outdoor Pollution Baseline", fontsize=13, fontweight="bold", color=NAVY)
axes = axes.flatten()

for i, col in enumerate(out_cols):
    axes[i].hist(df[col], bins=40, color=BLUE, edgecolor="white")
    axes[i].set_title(col)
    axes[i].set_xlabel("Concentration")

# O3 vs NO2 scatter
axes[5].scatter(df.outdoor_no2, df.outdoor_o3, alpha=0.1, s=5, color=NAVY)
m, b, *_ = stats.linregress(df.outdoor_no2, df.outdoor_o3)
xs = np.linspace(df.outdoor_no2.min(), df.outdoor_no2.max(), 100)
axes[5].plot(xs, m*xs + b, color=RED, linewidth=2, label=f"r = {o3_no2_r:.3f}")
axes[5].set_title("O3 vs NO2 (expect negative)")
axes[5].set_xlabel("outdoor_no2")
axes[5].set_ylabel("outdoor_o3")
axes[5].legend()

plt.tight_layout()
save(fig, "04_outdoor_baseline.png")

# ---------------------------------------
# SECTION 5 – INDOOR ACTIVITY EFFECTS
# ---------------------------------------
print("[5/12] Indoor Activity Effects")
log("SECTION 5 – INDOOR ACTIVITY EFFECTS", title=True)

poll_rates = ["pm25_rate", "pm10_rate", "no2_rate"]
cook    = df[df.cooking_event == True]
no_cook = df[df.cooking_event == False]
gas     = df[df.gas_cooking_or_gas_hob == 1]
elec    = df[(df.cooking_event == True) & (df.gas_cooking_or_gas_hob == 0)]

for p in poll_rates:
    cm  = cook[p].mean()
    ncm = no_cook[p].mean()
    ratio = cm / ncm
    flag = "✅" if ratio >= 1.2 else "⚠ weak"
    log(f"{p}: no_cooking={ncm:.3f}  cooking={cm:.3f}  ratio={ratio:.2f}×  {flag}")

t, p_val = stats.ttest_ind(gas.no2_rate, elec.no2_rate)
log(f"\nGas NO2 mean  : {gas.no2_rate.mean():.3f}")
log(f"Electric NO2  : {elec.no2_rate.mean():.3f}")
log(f"t-test p-value: {p_val:.4f}  {status_badge(p_val < 0.05)}")
log("⚠ Gas vs electric NO2 difference NOT statistically significant (p=0.39)")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Section 5 – Indoor Activity Effects", fontsize=13, fontweight="bold", color=NAVY)

for ax, p in zip(axes, poll_rates):
    means  = [no_cook[p].mean(), cook[p].mean()]
    errors = [no_cook[p].std() / np.sqrt(len(no_cook)),
              cook[p].std()    / np.sqrt(len(cook))]
    bars = ax.bar(["No Cooking", "Cooking"], means,
                  color=[GREEN, RED], yerr=errors, capsize=5)
    ax.set_title(p)
    ax.set_ylabel("Mean Rate")
    ratio = means[1] / means[0]
    ax.text(0.5, max(means)*1.05, f"ratio: {ratio:.2f}×",
            ha="center", transform=ax.get_xaxis_transform(), fontsize=9, color=NAVY)

plt.tight_layout()
save(fig, "05_activity_effects.png")

# gas vs electric subplot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Section 5b – Gas vs Electric Cooking (NO2)", fontsize=13, fontweight="bold", color=NAVY)

groups = {"No cooking": df[df.cooking_event==False],
          "Electric":   elec,
          "Gas":        gas}

for ax, p in zip(axes, poll_rates):
    means = [g[p].mean() for g in groups.values()]
    ax.bar(groups.keys(), means, color=[GREY, GREEN, RED])
    ax.set_title(p)
    ax.set_ylabel("Mean Rate")
    ax.tick_params(axis="x", rotation=10)

plt.tight_layout()
save(fig, "05b_gas_vs_electric.png")

# ---------------------------------------
# SECTION 6 – VENTILATION EFFECTS
# ---------------------------------------
print("[6/12] Ventilation Effects")
log("SECTION 6 – VENTILATION EFFECTS", title=True)

def matched_binary_effect(data, flag_col, rate_cols, group_cols=None, min_n_per_state=5):
    """
    Compare pollutant rates for flag_col == 1 vs flag_col == 0 within similar records.

    Why this is needed:
    A whole-dataset comparison can be misleading because open-window/fan-on rows may also
    contain higher-emission activities, longer cooking durations, or gas cooking. This
    function first groups by similar activity/context columns, then compares on vs off
    only inside groups that contain both states.
    """
    group_cols = [c for c in (group_cols or []) if c in data.columns]
    needed = [flag_col] + rate_cols + group_cols
    d = data[needed].dropna().copy()

    if d.empty or d[flag_col].nunique() < 2:
        return {}

    rows = []
    if group_cols:
        iterator = d.groupby(group_cols, dropna=False)
    else:
        iterator = [(("all",), d)]

    for group_key, g in iterator:
        off = g[g[flag_col] == 0]
        on = g[g[flag_col] == 1]

        if len(off) < min_n_per_state or len(on) < min_n_per_state:
            continue

        row = {
            "group": group_key if isinstance(group_key, tuple) else (group_key,),
            "n_off": len(off),
            "n_on": len(on),
            "weight": min(len(off), len(on)),
        }

        for col in rate_cols:
            off_mean = off[col].mean()
            on_mean = on[col].mean()
            row[f"{col}_off"] = off_mean
            row[f"{col}_on"] = on_mean
            row[f"{col}_diff"] = on_mean - off_mean
            row[f"{col}_pct_change"] = np.nan if off_mean == 0 else ((on_mean - off_mean) / off_mean) * 100

        rows.append(row)

    if not rows:
        return {}

    matched = pd.DataFrame(rows)
    summary = {}

    for col in rate_cols:
        w = matched["weight"]
        off_mean = np.average(matched[f"{col}_off"], weights=w)
        on_mean = np.average(matched[f"{col}_on"], weights=w)
        diff = on_mean - off_mean
        pct_change = np.nan if off_mean == 0 else (diff / off_mean) * 100
        n_groups_lower = int((matched[f"{col}_diff"] < 0).sum())
        n_groups_total = int(matched[f"{col}_diff"].notna().sum())

        summary[col] = {
            "off_mean": off_mean,
            "on_mean": on_mean,
            "diff": diff,
            "pct_change": pct_change,
            "n_groups_lower": n_groups_lower,
            "n_groups_total": n_groups_total,
            "matched_rows": int(matched["n_off"].sum() + matched["n_on"].sum()),
        }

    return {"summary": summary, "matched_groups": matched, "group_cols": group_cols}


# Create duration bands so rows are compared with similar cooking/activity intensity.
if "cooking_duration_min" in df.columns:
    df["_cooking_duration_band"] = pd.cut(
        df["cooking_duration_min"].fillna(0),
        bins=[-0.1, 0, 15, 30, 60, 120, np.inf],
        labels=["0", "1-15", "16-30", "31-60", "61-120", "120+"]
    )

window_group_cols = [
    "activity_type",
    "cooking_event",
    "gas_cooking_or_gas_hob",
    "cooking_method",
    "_cooking_duration_band",
]

# For extractor fan, compare mainly within cooking-like records and hold window state constant.
fan_group_cols = [
    "activity_type",
    "cooking_event",
    "gas_cooking_or_gas_hob",
    "cooking_method",
    "_cooking_duration_band",
    "window_open",
]

window_effect = matched_binary_effect(
    df,
    flag_col="window_open",
    rate_cols=poll_rates,
    group_cols=window_group_cols,
    min_n_per_state=5,
)

fan_base = df.copy()
if "cooking_event" in fan_base.columns:
    fan_base = fan_base[fan_base["cooking_event"] == True].copy()

fan_effect = matched_binary_effect(
    fan_base,
    flag_col="extractor_fan_on",
    rate_cols=poll_rates,
    group_cols=fan_group_cols,
    min_n_per_state=5,
)

log("Matched ventilation checks compare ON vs OFF only within similar activity/context groups.")
log(f"Window grouping columns used: {window_effect.get('group_cols', []) if window_effect else 'No valid matched groups'}")
log(f"Extractor grouping columns used: {fan_effect.get('group_cols', []) if fan_effect else 'No valid matched groups'}")

for p in poll_rates:
    log(f"{p}:")

    if window_effect:
        s = window_effect["summary"][p]
        w_ok = s["on_mean"] < s["off_mean"]
        log(
            f"  Window matched: closed={s['off_mean']:.3f}  open={s['on_mean']:.3f}  "
            f"change={s['pct_change']:.1f}%  groups_lower={s['n_groups_lower']}/{s['n_groups_total']}  "
            f"{'↓ ✅' if w_ok else '↑ ❌ REVERSED'}"
        )
    else:
        log("  Window matched: not enough comparable open/closed rows within similar activity groups")

    if fan_effect:
        s = fan_effect["summary"][p]
        e_ok = s["on_mean"] < s["off_mean"]
        log(
            f"  Extractor matched: off={s['off_mean']:.3f}  on={s['on_mean']:.3f}  "
            f"change={s['pct_change']:.1f}%  groups_lower={s['n_groups_lower']}/{s['n_groups_total']}  "
            f"{'↓ ✅' if e_ok else '↑ ❌'}"
        )
    else:
        log("  Extractor matched: not enough comparable fan-on/fan-off cooking rows within similar groups")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Section 6 – Matched Ventilation Effects", fontsize=13, fontweight="bold", color=NAVY)

for j, p in enumerate(poll_rates):
    # window: matched activity/context comparison
    if window_effect:
        s = window_effect["summary"][p]
        wc, wo = s["off_mean"], s["on_mean"]
        col = RED if wo > wc else GREEN
        axes[0][j].bar(["Closed\nmatched", "Open\nmatched"], [wc, wo], color=[GREEN, col])
        axes[0][j].set_title(f"Window Open – {p}")
        axes[0][j].set_ylabel("Mean Rate")
        axes[0][j].text(0.5, 0.92, f"{s['pct_change']:.1f}%", transform=axes[0][j].transAxes,
                        ha="center", fontsize=9, color=NAVY)
        if wo > wc:
            axes[0][j].set_facecolor("#FCE4D6")
            axes[0][j].annotate("⚠ HIGHER when open", xy=(0.5, 0.82),
                                xycoords="axes fraction", ha="center",
                                fontsize=8, color="red")
    else:
        axes[0][j].text(0.5, 0.5, "Not enough\nmatched groups", ha="center", va="center")
        axes[0][j].set_title(f"Window Open – {p}")

    # extractor: matched cooking/context comparison
    if fan_effect:
        s = fan_effect["summary"][p]
        eof, eon = s["off_mean"], s["on_mean"]
        col = RED if eon > eof else GREEN
        axes[1][j].bar(["Off\nmatched", "On\nmatched"], [eof, eon], color=[GREEN, col])
        axes[1][j].set_title(f"Extractor Fan – {p}")
        axes[1][j].set_ylabel("Mean Rate")
        axes[1][j].text(0.5, 0.92, f"{s['pct_change']:.1f}%", transform=axes[1][j].transAxes,
                        ha="center", fontsize=9, color=NAVY)
        if eon > eof:
            axes[1][j].set_facecolor("#FCE4D6")
            axes[1][j].annotate("⚠ HIGHER when on", xy=(0.5, 0.82),
                                xycoords="axes fraction", ha="center",
                                fontsize=8, color="red")
    else:
        axes[1][j].text(0.5, 0.5, "Not enough\nmatched groups", ha="center", va="center")
        axes[1][j].set_title(f"Extractor Fan – {p}")

plt.tight_layout()
save(fig, "06_ventilation_matched.png")

# Optional detailed matched-group output for audit/debugging
if window_effect:
    window_effect["matched_groups"].to_csv(OUT_DIR / "06_window_matched_group_details.csv", index=False)
if fan_effect:
    fan_effect["matched_groups"].to_csv(OUT_DIR / "06_extractor_matched_group_details.csv", index=False)


# ---------------------------------------
# SECTION 7 – DISTRIBUTIONS
# ---------------------------------------   
print("[7/12] Distributions")
log("SECTION 7 – DISTRIBUTIONS", title=True)

for p in poll_rates:
    sk = stats.skew(df[p].dropna())
    log(f"{p}: skew={sk:.2f}  min={df[p].min():.3f}  max={df[p].max():.3f}  "
        f"mean={df[p].mean():.3f}  {'✅ right-skewed' if sk > 0 else '❌'}")

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Section 7 – Pollutant Distributions", fontsize=13, fontweight="bold", color=NAVY)

for j, p in enumerate(poll_rates):
    sk = stats.skew(df[p].dropna())
    # histogram
    axes[0][j].hist(df[p], bins=50, color=BLUE, edgecolor="white")
    axes[0][j].set_title(f"{p}\nskew = {sk:.2f}")
    axes[0][j].set_xlabel("Rate")

    # box plots split by cooking
    data = [df[df.cooking_event==False][p].dropna(),
            df[df.cooking_event==True][p].dropna()]
    bp = axes[1][j].boxplot(data, labels=["No Cooking", "Cooking"],
                             patch_artist=True, showfliers=False)
    for patch, color in zip(bp["boxes"], [GREEN, RED]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    axes[1][j].set_title(f"{p} by Cooking Event")

plt.tight_layout()
save(fig, "07_distributions.png")

# ---------------------------------------
# SECTION 8 – CORRELATIONS
#  ---------------------------------------
print("[8/12] Correlations")
log("SECTION 8 – CORRELATIONS", title=True)

corr_cols = ["pm25_rate", "pm10_rate", "no2_rate",
             "outdoor_o3", "outdoor_no2", "gas_cooking_or_gas_hob",
             "window_open", "extractor_fan_on", "wood_burning_event",
             "smoking_vaping_exposure"]
corr = df[corr_cols].corr()

r_pm   = corr.loc["pm25_rate",  "pm10_rate"]
r_no2g = corr.loc["no2_rate",   "gas_cooking_or_gas_hob"]
r_o3n  = corr.loc["outdoor_o3", "outdoor_no2"]

log(f"PM2.5 ~ PM10        : {r_pm:.3f}  {'✅' if r_pm > 0.8 else '⚠ unexpectedly perfect — suspicious' if r_pm > 0.99 else '⚠'}")
log(f"NO2   ~ gas cooking : {r_no2g:.3f}  {'✅' if r_no2g > 0.1 else '❌ very weak'}")
log(f"O3    ~ NO2         : {r_o3n:.3f}   {'✅ negative' if r_o3n < 0 else '❌'}")
if r_pm > 0.99:
    log("⚠ PM2.5–PM10 correlation is 1.000 — may be generated from same source without noise")

fig, ax = plt.subplots(figsize=(10, 8))
fig.suptitle("Section 8 – Correlation Matrix", fontsize=13, fontweight="bold", color=NAVY)

im = ax.imshow(corr.values, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(corr_cols, fontsize=8)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.values[i,j]:.2f}", ha="center", va="center",
                fontsize=7, color="black")
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
save(fig, "08_correlations.png")

# ---------------------------------------
# SECTION 9 – TIME PATTERNS
# ---------------------------------------
print("[9/12] Time Patterns")
log("SECTION 9 – TIME PATTERNS", title=True)

hourly_cook = df.groupby("hour")["cooking_event"].mean()
peak_cook   = int(hourly_cook.idxmax())
hourly_o3   = df.groupby("hour")["outdoor_o3"].mean()
peak_o3     = int(hourly_o3.idxmax())

log(f"Peak cooking hour : {peak_cook}:00  (rate={hourly_cook[peak_cook]:.2%})")
log(f"  ⚠ No clear evening peak — rate is roughly flat across all hours")
log(f"Peak O3 hour      : {peak_o3}:00  (mean={hourly_o3[peak_o3]:.2f})  ✅ afternoon plausible")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Section 9 – Time Patterns", fontsize=13, fontweight="bold", color=NAVY)

# cooking rate by hour
axes[0].bar(hourly_cook.index, hourly_cook.values, color=RED, alpha=0.8)
axes[0].axhspan(0.30, 0.45, color=GREEN, alpha=0.15, label="Expected evening peak zone")
axes[0].set_title("Cooking Event Rate by Hour")
axes[0].set_xlabel("Hour of Day")
axes[0].set_ylabel("Fraction of sessions with cooking")
axes[0].legend(fontsize=7)
axes[0].set_xticks(range(0, 24, 2))

# O3 by hour
axes[1].plot(hourly_o3.index, hourly_o3.values, color=AMBER, marker="o", linewidth=2)
axes[1].fill_between(hourly_o3.index, hourly_o3.values, alpha=0.2, color=AMBER)
axes[1].axvline(peak_o3, color=RED, linestyle="--", label=f"Peak at {peak_o3}:00")
axes[1].set_title("Mean Outdoor O3 by Hour")
axes[1].set_xlabel("Hour of Day")
axes[1].set_ylabel("O3 concentration")
axes[1].legend(fontsize=8)
axes[1].set_xticks(range(0, 24, 2))

# NO2 by hour
hourly_no2 = df.groupby("hour")["outdoor_no2"].mean()
axes[2].plot(hourly_no2.index, hourly_no2.values, color=NAVY, marker="o", linewidth=2)
axes[2].fill_between(hourly_no2.index, hourly_no2.values, alpha=0.2, color=NAVY)
axes[2].set_title("Mean Outdoor NO2 by Hour")
axes[2].set_xlabel("Hour of Day")
axes[2].set_ylabel("NO2 concentration")
axes[2].set_xticks(range(0, 24, 2))

plt.tight_layout()
save(fig, "09_time_patterns.png")

# ---------------------------------------
# SECTION 10 – DOSE VARIABLES
# ---------------------------------------
print("[10/12] Dose Variables")
log("SECTION 10 – DOSE VARIABLES", title=True)

df["pm25_dose"] = df["pm25_rate"] * df["duration_min"]
df["pm10_dose"] = df["pm10_rate"] * df["duration_min"]
df["no2_dose"]  = df["no2_rate"]  * df["duration_min"]

for d, p in [("pm25_dose","pm25_rate"),("pm10_dose","pm10_rate"),("no2_dose","no2_rate")]:
    r = df[d].corr(df["duration_min"])
    log(f"{d} ~ duration : r={r:.3f}  {'✅' if r > 0 else '❌'}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Section 10 – Dose Variables", fontsize=13, fontweight="bold", color=NAVY)

for ax, (d, label) in zip(axes, [("pm25_dose","PM2.5"),("pm10_dose","PM10"),("no2_dose","NO2")]):
    sample = df.sample(min(2000, len(df)), random_state=42)
    ax.scatter(sample["duration_min"], sample[d], alpha=0.3, s=8, color=BLUE)
    m, b, *_ = stats.linregress(df["duration_min"], df[d])
    xs = np.linspace(df["duration_min"].min(), df["duration_min"].max(), 100)
    r  = df[d].corr(df["duration_min"])
    ax.plot(xs, m*xs + b, color=RED, linewidth=2, label=f"r={r:.2f}")
    ax.set_title(f"{label} Dose vs Duration")
    ax.set_xlabel("duration_min")
    ax.set_ylabel(f"{d}")
    ax.legend(fontsize=8)

plt.tight_layout()
save(fig, "10_dose_variables.png")

# ---------------------------------------
# SECTION 11 – OUTLIERS
# ---------------------------------------
print("[11/12] Outliers")
log("SECTION 11 – OUTLIERS", title=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Section 11 – Outlier Analysis", fontsize=13, fontweight="bold", color=NAVY)

for ax, p in zip(axes, poll_rates):
    q99 = df[p].quantile(0.99)
    out = df[df[p] > q99]
    log(f"\n{p} >99th pct ({q99:.3f}): n={len(out)}")
    if len(out) > 0:
        log(f"  cooking      : {out.cooking_event.mean():.0%}")
        log(f"  wood burning : {out.wood_burning_event.mean():.0%}")
        log(f"  smoking/vaping: {out.smoking_vaping_exposure.mean():.0%}")

    # box plot
    ax.boxplot(df[p].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor=BLUE, alpha=0.5),
               flierprops=dict(marker="o", markersize=2, color=RED, alpha=0.4))
    ax.set_title(f"{p}\n99th pct = {q99:.2f}")
    ax.set_ylabel("Rate")

plt.tight_layout()
save(fig, "11_outliers.png")

# ---------------------------------------
# SECTION 12 – LOGICAL CONSISTENCY
# ---------------------------------------
print("[12/12] Logical Consistency")
log("SECTION 12 – LOGICAL CONSISTENCY", title=True)

# No cooking event → cooking_method = 'none'
no_cook_methods = df[df.cooking_event == False]["cooking_method"].value_counts()
ok_no_method    = set(no_cook_methods.index) == {"none"}
log(f"No-cooking rows have method='none': {status_badge(ok_no_method)}")

# PM10_rate >= PM2.5_rate
pm_viol = (df["pm10_rate"] < df["pm25_rate"]).sum()
log(f"PM10_rate < PM2.5_rate violations: {pm_viol}  {status_badge(pm_viol==0)}")

# cooking_duration_min when no event
no_cook_dur_nonzero = (df[df.cooking_event==False]["cooking_duration_min"] > 0).sum()
log(f"cooking_duration > 0 when no event: {no_cook_dur_nonzero}  {status_badge(no_cook_dur_nonzero==0)}")

# Oven in emission factors but not dataset
oven_in_ef = "oven" in ef.cooking_method.values
oven_in_df = "oven" in df.cooking_method.values
log(f"'oven' in emission factors: {oven_in_ef}  |  'oven' in dataset: {oven_in_df}  ⚠ mismatch")

# NO2 for electric vs gas
elec_no2 = df[(df.cooking_event==True) & (df.gas_cooking_or_gas_hob==0)]["no2_rate"].mean()
gas_no2  = df[df.gas_cooking_or_gas_hob==1]["no2_rate"].mean()
log(f"Electric cooking NO2: {elec_no2:.3f}  |  Gas cooking NO2: {gas_no2:.3f}")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Section 12 – Logical Consistency", fontsize=13, fontweight="bold", color=NAVY)
axes = axes.flatten()

# 1. cooking method when no event
axes[0].bar(no_cook_methods.index, no_cook_methods.values, color=GREEN)
axes[0].set_title("Cooking Method when cooking_event=False\n✅ Only 'none'")
axes[0].set_ylabel("Count")

# 2. PM10 vs PM2.5 scatter
sample = df.sample(min(3000, len(df)), random_state=1)
axes[1].scatter(sample["pm25_rate"], sample["pm10_rate"], alpha=0.2, s=5, color=BLUE)
mx = max(df.pm25_rate.max(), df.pm10_rate.max())
axes[1].plot([0, mx], [0, mx], color=RED, linestyle="--", label="PM10=PM2.5 line")
axes[1].set_title(f"PM10_rate vs PM2.5_rate\n✅ {pm_viol} violations")
axes[1].set_xlabel("pm25_rate")
axes[1].set_ylabel("pm10_rate")
axes[1].legend(fontsize=7)

# 3. cooking_duration when no event
dur_no_cook = df[df.cooking_event==False]["cooking_duration_min"]
axes[2].bar(["cooking_event=False"], [dur_no_cook.max()], color=GREEN)
axes[2].set_title(f"Max cooking_duration when no event: {dur_no_cook.max()}\n✅ Always 0")
axes[2].set_ylabel("Max cooking_duration_min")

# 4. Summary checklist as text
checks = [
    ("No cooking → method = 'none'",      ok_no_method),
    ("PM10_rate ≥ PM2.5_rate always",      pm_viol == 0),
    ("cooking_duration = 0 when no event", no_cook_dur_nonzero == 0),
    ("Gas NO2 > Electric NO2",             gas_no2 > elec_no2),
    ("'oven' in dataset (matches EF)",     oven_in_df),
]
axes[3].axis("off")
y = 0.9
for label, ok in checks:
    symbol = "✅" if ok else "❌"
    color  = "green" if ok else "red"
    axes[3].text(0.05, y, f"{symbol}  {label}", transform=axes[3].transAxes,
                 fontsize=10, color=color, va="top")
    y -= 0.16
axes[3].set_title("Logical Consistency Checklist", pad=10)

plt.tight_layout()
save(fig, "12_logical_consistency.png")

# ---------------------------------------
# EMISSION FACTORS – bonus check
# ---------------------------------------
log("EMISSION FACTORS FILE", title=True)
log(ef.to_string(index=False))
log("\nMethods in EF   : " + str(sorted(ef.cooking_method.unique())))
log("Methods in dataset: " + str(sorted(df.cooking_method.unique())))
log("⚠ 'oven' present in EF but missing from dataset")

# ---------------------------------------
# SUMMARY DASHBOARD
# ---------------------------------------


def make_check(label, status, detail=""):
    return (label, status, detail)

def classify_binary(pass_condition, warn_condition=None):
    if pass_condition:
        return "PASS"
    elif warn_condition is None or warn_condition:
        return "WARN"
    return "FAIL"

def build_summary_checks(df, ef):
    checks = []

    # 1. row count / duplicates
    dupes = df.duplicated().sum()
    row_label = f"Row count = {len(df):,} / Duplicates = {dupes}"
    row_status = "PASS" if dupes == 0 else "FAIL"
    row_detail = "" if dupes == 0 else f"{dupes} duplicate rows found"
    checks.append(make_check(row_label, row_status, row_detail))

    # 2. time_of_day missing
    tod_missing = df["time_of_day"].isna().sum() if "time_of_day" in df.columns else len(df)
    if tod_missing == 0:
        checks.append(make_check("time_of_day has no missing values", "PASS", ""))
    else:
        detail = f"{tod_missing} missing"
        if "hour" in df.columns:
            missing_hours = sorted(df.loc[df["time_of_day"].isna(), "hour"].dropna().unique().tolist())
            if missing_hours:
                detail += f" at hour(s) {missing_hours}"
        checks.append(make_check(f"time_of_day missing = {tod_missing}", "WARN", detail))

    # 3. duration range
    dur_min = df["duration_min"].min()
    dur_max = df["duration_min"].max()
    if dur_max >= 120:
        status = "PASS"
        detail = f"range = {dur_min}–{dur_max}"
    elif dur_max >= 60:
        status = "WARN"
        detail = f"range = {dur_min}–{dur_max} (below 120)"
    else:
        status = "FAIL"
        detail = f"range = {dur_min}–{dur_max}"
    checks.append(make_check("Duration range reaches expected upper bound", status, detail))

    # 4. all 24 hours covered
    if "hour" not in df.columns:
        df["hour"] = pd.to_datetime(df["start_time_local"]).dt.hour
    n_hours = df["hour"].nunique()
    missing_hours = sorted(set(range(24)) - set(df["hour"].dropna().unique()))
    if n_hours == 24:
        checks.append(make_check("All 24 hours covered", "PASS", ""))
    elif n_hours >= 20:
        checks.append(make_check("All 24 hours covered", "WARN", f"Missing hours: {missing_hours}"))
    else:
        checks.append(make_check("All 24 hours covered", "FAIL", f"Only {n_hours}/24 hours represented"))

    # 5. outdoor PM10 >= PM2.5
    outdoor_pm10_col = "outdoor_pm10" if "outdoor_pm10" in df.columns else "pm10_avg"
    outdoor_pm25_col = "outdoor_pm25" if "outdoor_pm25" in df.columns else "pm25_avg"
    if outdoor_pm10_col in df.columns and outdoor_pm25_col in df.columns:
        bad = (df[outdoor_pm10_col] < df[outdoor_pm25_col]).sum()
        checks.append(make_check(
            "Outdoor PM10 ≥ PM2.5 always",
            "PASS" if bad == 0 else "FAIL",
            "" if bad == 0 else f"{bad} rows violate PM10 ≥ PM2.5"
        ))

    # 6. no negative outdoor values
    outdoor_cols = [c for c in df.columns if c in ["no2_avg", "pm25_avg", "pm10_avg", "o3_avg", "so2_avg",
                                                   "outdoor_no2", "outdoor_pm25", "outdoor_pm10", "outdoor_o3", "outdoor_so2"]]
    neg_count = 0
    if outdoor_cols:
        neg_count = int((df[outdoor_cols] < 0).sum().sum())
    checks.append(make_check(
        "No negative outdoor values",
        "PASS" if neg_count == 0 else "FAIL",
        "" if neg_count == 0 else f"{neg_count} negative values found"
    ))

    # 7. O3 ~ NO2 negative correlation
    if "o3_avg" in df.columns and "no2_avg" in df.columns:
        o3_no2_r = df["o3_avg"].corr(df["no2_avg"])
        if pd.isna(o3_no2_r):
            status, detail = "FAIL", "Correlation not computable"
        elif o3_no2_r < -0.1:
            status, detail = "PASS", f"r = {o3_no2_r:.3f}"
        elif o3_no2_r < 0:
            status, detail = "WARN", f"r = {o3_no2_r:.3f} (weak negative)"
        else:
            status, detail = "FAIL", f"r = {o3_no2_r:.3f} (not negative)"
        checks.append(make_check("O3 ~ NO2 negative correlation", status, detail))

    # 8. cooking raises pollutants
    needed = {"activity_type", "pm25_rate", "pm10_rate", "no2_rate"}
    if needed.issubset(df.columns):
        cook_mask = df["activity_type"].astype(str).str.lower().eq("cooking")
        noncook_mask = ~cook_mask
        if cook_mask.sum() > 0 and noncook_mask.sum() >= 0:
            ratios = {}
            for col in ["pm25_rate", "pm10_rate", "no2_rate"]:
                base = df.loc[noncook_mask, col].mean()
                cook = df.loc[cook_mask, col].mean()
                ratios[col] = np.nan if base == 0 else cook / base
            min_ratio = np.nanmin(list(ratios.values()))
            detail = " / ".join([f"{k}: {v:.2f}×" for k, v in ratios.items()])
            if min_ratio >= 2.0:
                status = "PASS"
            elif min_ratio >= 1.2:
                status = "WARN"
            else:
                status = "FAIL"
            checks.append(make_check("Cooking raises PM2.5/PM10/NO2", status, detail))

    # 9. gas cooking NO2 > electric
    needed = {"activity_type", "gas_cooking_or_gas_hob", "no2_rate"}
    if needed.issubset(df.columns):
        cook_df = df[df["activity_type"].astype(str).str.lower() == "cooking"].copy()
        gas = cook_df.loc[cook_df["gas_cooking_or_gas_hob"] == 1, "no2_rate"]
        elec = cook_df.loc[cook_df["gas_cooking_or_gas_hob"] == 0, "no2_rate"]
        if len(gas) > 1 and len(elec) > 1:
            _, p = stats.ttest_ind(gas, elec, equal_var=False, nan_policy="omit")
            gas_mean = gas.mean()
            elec_mean = elec.mean()
            if gas_mean > elec_mean and p < 0.05:
                status = "PASS"
            elif gas_mean > elec_mean:
                status = "WARN"
            else:
                status = "FAIL"
            detail = f"gas={gas_mean:.2f}, electric={elec_mean:.2f}, p={p:.3g}"
            checks.append(make_check("Gas cooking NO2 > electric (sig.)", status, detail))

    # 10. window open lowers pollutants
    # Uses matched/grouped comparison from Section 6 instead of whole-dataset means.
    if "window_effect" in globals() and window_effect:
        lowers = 0
        detail_parts = []
        for col in ["pm25_rate", "pm10_rate", "no2_rate"]:
            s = window_effect["summary"][col]
            if s["on_mean"] < s["off_mean"]:
                lowers += 1
            detail_parts.append(
                f"{col}: open={s['on_mean']:.2f} vs closed={s['off_mean']:.2f} "
                f"({s['pct_change']:.1f}%, groups {s['n_groups_lower']}/{s['n_groups_total']})"
            )
        detail = " / ".join(detail_parts)
        if lowers == 3:
            status = "PASS"
        elif lowers >= 1:
            status = "WARN"
        else:
            status = "FAIL"
        checks.append(make_check("Window open ↓ pollutants, matched activities", status, detail))

    # 11. extractor fan lowers pollutants
    # Uses cooking-only matched/grouped comparison from Section 6.
    if "fan_effect" in globals() and fan_effect:
        lowers = 0
        detail_parts = []
        for col in ["pm25_rate", "pm10_rate", "no2_rate"]:
            s = fan_effect["summary"][col]
            if s["on_mean"] < s["off_mean"]:
                lowers += 1
            detail_parts.append(
                f"{col}: on={s['on_mean']:.2f} vs off={s['off_mean']:.2f} "
                f"({s['pct_change']:.1f}%, groups {s['n_groups_lower']}/{s['n_groups_total']})"
            )
        detail = " / ".join(detail_parts)
        if lowers == 3:
            status = "PASS"
        elif lowers >= 1:
            status = "WARN"
        else:
            status = "FAIL"
        checks.append(make_check("Extractor fan ↓ pollutants, matched cooking activities", status, detail))

    # 12. right-skewed distributions
    skew_cols = [c for c in ["pm25_rate", "pm10_rate", "no2_rate"] if c in df.columns]
    if skew_cols:
        skews = df[skew_cols].skew(numeric_only=True)
        positive_skews = (skews > 0.5).sum()
        detail = " / ".join([f"{k}: {v:.2f}" for k, v in skews.items()])
        if positive_skews == len(skew_cols):
            status = "PASS"
        elif positive_skews >= 1:
            status = "WARN"
        else:
            status = "FAIL"
        checks.append(make_check("Right-skewed distributions", status, detail))

    # 13. PM2.5–PM10 correlation
    if {"pm25_rate", "pm10_rate"}.issubset(df.columns):
        r = df["pm25_rate"].corr(df["pm10_rate"])
        if pd.isna(r):
            status, detail = "FAIL", "Correlation not computable"
        elif 0.7 <= r < 0.98:
            status, detail = "PASS", f"r = {r:.3f}"
        elif 0.98 <= r <= 1.0:
            status, detail = "WARN", f"r = {r:.3f} — suspiciously perfect"
        else:
            status, detail = "WARN", f"r = {r:.3f}"
        checks.append(make_check("PM2.5–PM10 correlation", status, detail))

    # 14. NO2 ~ gas cooking correlation
    if {"no2_rate", "gas_cooking_or_gas_hob"}.issubset(df.columns):
        r = df["no2_rate"].corr(df["gas_cooking_or_gas_hob"])
        if pd.isna(r):
            status, detail = "FAIL", "Correlation not computable"
        elif r >= 0.15:
            status, detail = "PASS", f"r = {r:.3f}"
        elif r > 0.05:
            status, detail = "WARN", f"r = {r:.3f}"
        else:
            status, detail = "FAIL", f"r = {r:.3f}"
        checks.append(make_check("NO2 ~ gas cooking correlation", status, detail))

    # 15. evening cooking peak
    if {"hour", "activity_type"}.issubset(df.columns):
        cook_hour = df.assign(is_cooking=df["activity_type"].astype(str).str.lower().eq("cooking")) \
                      .groupby("hour")["is_cooking"].mean()
        evening_peak = cook_hour.loc[17:20].mean()
        non_evening = cook_hour.drop(index=[h for h in range(17, 21) if h in cook_hour.index]).mean()
        peak_hour = int(cook_hour.idxmax())
        if evening_peak > non_evening * 1.2:
            status = "PASS"
        elif evening_peak > non_evening:
            status = "WARN"
        else:
            status = "FAIL"
        detail = f"peak at {peak_hour}:00"
        checks.append(make_check("Evening cooking peak", status, detail))

    # 16. midday/afternoon O3 peak
    if {"hour", "o3_avg"}.issubset(df.columns):
        o3_by_hour = df.groupby("hour")["o3_avg"].mean()
        peak_o3 = int(o3_by_hour.idxmax())
        if 12 <= peak_o3 <= 17:
            status = "PASS"
        elif 10 <= peak_o3 <= 19:
            status = "WARN"
        else:
            status = "FAIL"
        checks.append(make_check("Midday/afternoon O3 peak", status, f"Peak at {peak_o3}:00"))

    # 17. dose ~ duration positive
    dose_col = next((c for c in ["pm25_dose", "no2_dose", "pm10_dose"] if c in df.columns), None)
    if dose_col and "duration_min" in df.columns:
        r = df[dose_col].corr(df["duration_min"])
        if pd.isna(r):
            status, detail = "FAIL", "Correlation not computable"
        elif r > 0.15:
            status, detail = "PASS", f"r = {r:.3f}"
        elif r > 0:
            status, detail = "WARN", f"r = {r:.3f}"
        else:
            status, detail = "FAIL", f"r = {r:.3f}"
        checks.append(make_check("Dose ~ duration positive", status, detail))

    # 18. no emission when no cooking
    rate_cols = [c for c in ["pm25_rate", "pm10_rate", "no2_rate"] if c in df.columns]
    if "activity_type" in df.columns and rate_cols:
        noncook = df["activity_type"].astype(str).str.lower() != "cooking"
        means = df.loc[noncook, rate_cols].mean()
        detail = " / ".join([f"{k}: {v:.2f}" for k, v in means.items()])
        if (means < 0.05).all():
            status = "PASS"
        elif (means < 0.5).all():
            status = "WARN"
        else:
            status = "FAIL"
        checks.append(make_check("No emission when no cooking", status, detail))

    # 19. indoor PM10_rate ≥ PM2.5_rate
    if {"pm10_rate", "pm25_rate"}.issubset(df.columns):
        bad = (df["pm10_rate"] < df["pm25_rate"]).sum()
        checks.append(make_check(
            "Indoor PM10_rate ≥ PM2.5_rate",
            "PASS" if bad == 0 else "FAIL",
            "" if bad == 0 else f"{bad} rows violate PM10 ≥ PM2.5"
        ))

    # 20. oven present in both EF and data
    if "cooking_method" in df.columns and "cooking_method" in ef.columns:
        oven_in_ef = ef["cooking_method"].astype(str).str.lower().eq("oven").any()
        oven_in_df = df["cooking_method"].astype(str).str.lower().eq("oven").any()
        if oven_in_ef and oven_in_df:
            status, detail = "PASS", ""
        elif oven_in_ef and not oven_in_df:
            status, detail = "WARN", "Present in EF but absent in data"
        else:
            status, detail = "WARN", "Not found in EF"
        checks.append(make_check("'oven' in dataset (matches EF)", status, detail))

    # 21. motion_type diversity
    diversity_cols = [c for c in ["motion_type", "environment_type"] if c in df.columns]
    if diversity_cols:
        nunique = {c: df[c].nunique(dropna=True) for c in diversity_cols}
        detail = " / ".join([f"{k}: {v} unique" for k, v in nunique.items()])
        if all(v > 1 for v in nunique.values()):
            status = "PASS"
        elif any(v > 1 for v in nunique.values()):
            status = "WARN"
        else:
            status = "WARN"
        checks.append(make_check("motion_type / environment diversity", status, detail))
    
        # 22. cooking event completeness (no nulls when cooking)
    required_cols = [
        "cooking_method",
        "cooking_duration_min",
        "gas_cooking_or_gas_hob"
    ]

    if "cooking_event" in df.columns:
        cook_df = df[df["cooking_event"] == True]

        if len(cook_df) == 0:
            checks.append(make_check(
                "Cooking event completeness",
                "WARN",
                "No cooking events present"
            ))
        else:
            null_counts = cook_df[required_cols].isna().sum()
            total_nulls = int(null_counts.sum())

            if total_nulls == 0:
                checks.append(make_check(
                    "Cooking event completeness",
                    "PASS",
                    "No nulls in required fields"
                ))
            else:
                detail = ", ".join(
                    [f"{col}: {int(cnt)}" for col, cnt in null_counts.items() if cnt > 0]
                )

                # classify severity
                null_ratio = total_nulls / (len(cook_df) * len(required_cols))

                if null_ratio < 0.05:
                    status = "WARN"
                else:
                    status = "FAIL"

                checks.append(make_check(
                    "Cooking event completeness",
                    status,
                    detail
                ))

    return checks


summary_checks = build_summary_checks(df, ef)

color_map = {"PASS": GREEN, "WARN": AMBER, "FAIL": RED}
fig, ax = plt.subplots(figsize=(14, max(8, 0.5 * len(summary_checks))))
fig.patch.set_facecolor(LIGHT)
ax.axis("off")

fig.suptitle(
    "IAQ EDA – Validation Summary Dashboard",
    fontsize=16,
    fontweight="bold",
    color=NAVY,
    y=0.98
)

pass_n = sum(1 for _, s, _ in summary_checks if s == "PASS")
warn_n = sum(1 for _, s, _ in summary_checks if s == "WARN")
fail_n = sum(1 for _, s, _ in summary_checks if s == "FAIL")

legend_items = [
    Patch(color=GREEN, label=f"PASS ({pass_n})"),
    Patch(color=AMBER, label=f"WARN ({warn_n})"),
    Patch(color=RED,   label=f"FAIL ({fail_n})"),
]
ax.legend(handles=legend_items, loc="upper right", fontsize=11,
          framealpha=0.85, bbox_to_anchor=(0.98, 0.98))

n = len(summary_checks)
top = 0.94
bottom = 0.04
usable_h = top - bottom
row_h = usable_h / n

for i, (label, status, detail) in enumerate(summary_checks):
    y_pos = top - i * row_h
    col = color_map[status]

    ax.add_patch(plt.Rectangle(
        (0.01, y_pos - row_h * 0.85), 0.98, row_h * 0.9,
        color=col, alpha=0.18, transform=ax.transAxes, clip_on=False
    ))
    ax.add_patch(plt.Rectangle(
        (0.01, y_pos - row_h * 0.85), 0.012, row_h * 0.9,
        color=col, alpha=0.95, transform=ax.transAxes, clip_on=False
    ))

    ax.text(0.04, y_pos - row_h * 0.3, label,
            transform=ax.transAxes, fontsize=9, va="center", fontweight="bold")
    ax.text(0.60, y_pos - row_h * 0.3, status,
            transform=ax.transAxes, fontsize=9, va="center",
            color=col, fontweight="bold")
    ax.text(0.70, y_pos - row_h * 0.3, detail,
            transform=ax.transAxes, fontsize=8, va="center",
            color="#555555", style="italic")

save(fig, "00_summary_dashboard.png")
# -----------------------
# TEXT REPORT
# -----------------------
report_path = OUT_DIR / "eda_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("IAQ SYNTHETIC DATASET – EDA REPORT\n")
    f.write("="*60 + "\n")
    f.write("\n".join(REPORT_LINES))
print(f"\n  Saved → {report_path}")

# -----------------------
# DONE
# -----------------------
print(f"""
╔══════════════════════════════════════════════════════╗
║  EDA complete!  Results in:  eda_report/             ║
║                                                      ║
║  00_summary_dashboard.png  ← start here              ║
║  01_data_quality.png                                 ║
║  02_time_duration.png                                ║
║  03_categorical.png                                  ║
║  04_outdoor_baseline.png                             ║
║  05_activity_effects.png                             ║
║  05b_gas_vs_electric.png                             ║
║  06_ventilation.png                                  ║
║  07_distributions.png                                ║
║  08_correlations.png                                 ║
║  09_time_patterns.png                                ║
║  10_dose_variables.png                               ║
║  11_outliers.png                                     ║
║  12_logical_consistency.png                          ║
║  eda_report.txt            ← full text log           ║
╚══════════════════════════════════════════════════════╝
""")